# Phase 1 - Notebook 2: Single-feature vs MTR error (train-only, Spearman-only)

**Goal.** Rank every individual candidate feature by how strongly it is *monotonically* associated with MTR error on the **AV2 train split**. The sole metric is **Spearman correlation** against `mtr_minfde6`.

**Why this is simpler than before (and why that is correct for Phase 1):**
- Log1p / raw / Pearson / rank-on-rank versions are redundant under Spearman, so we score each base numeric feature once.
- Binary-cutoff metrics (`top_20_capture`, `auroc_hard`, `ndcg_at_20`, `hard_vs_easy_uplift`) all depend on an arbitrary 20% threshold and were dropped.
- Phase 1's downstream consumer is a **training-time** reweighting / resampling strategy on train. Train-only evaluation is the right surface here; a train->val generalisation check is deferred to Phase 2.

**Outputs:**
- `artifacts/phase1/metrics/nb2_feature_ranking_train.csv` - single clean ranked table (one row per base feature).
- `artifacts/phase1/nb2_verdict.json` with `{best_feature, top_5_features, top_k_by_spearman, handcrafted_component_spearman}` where the last is the individual Spearman of each of the five handcrafted-composite components, reused by NB3 and the report.
- `artifacts/phase1/figures/nb2_top_features.png` - horizontal bar chart of top-20 features by `|Spearman|`.

**Verdict.** **PASS** iff at least one feature has `|Spearman| >= 0.15` on train. Numbers are shown for sanity only; the actual GO/PIVOT/STOP decision lives in NB3.

In [ ]:
from __future__ import annotations
import sys, json, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SRC_ROOT = REPO_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from tailrisk_mp.feature_analysis import (
    candidate_feature_columns, ID_COLUMNS, ERROR_COLUMNS,
)

ARTIFACTS = REPO_ROOT / 'artifacts' / 'phase1'
TABLES    = ARTIFACTS / 'tables'
METRICS   = ARTIFACTS / 'metrics'
FIG_DIR   = ARTIFACTS / 'figures'
for _d in (TABLES, METRICS, FIG_DIR):
    _d.mkdir(parents=True, exist_ok=True)

warnings.filterwarnings('ignore', category=FutureWarning)

PRIMARY_ERROR = 'mtr_minfde6'

def _load(parquet, csv_fallback):
    if parquet.exists(): return pd.read_parquet(parquet)
    if csv_fallback.exists(): return pd.read_csv(csv_fallback)
    return pd.DataFrame()

# Train-only: NB1's canonical output is the source of truth.
train = _load(ARTIFACTS / 'cleaned_train_with_style.parquet',
              ARTIFACTS / 'cleaned_train_with_style.csv')

if train.empty:
    raise FileNotFoundError(
        'cleaned_train_with_style.parquet not found under artifacts/phase1/. '
        'Run phase1_style_validation.ipynb first to produce it.'
    )

if PRIMARY_ERROR not in train.columns:
    raise KeyError(f'{PRIMARY_ERROR} missing from train; re-run run_difficulty_analysis to merge MTR errors.')

print('train:', train.shape)
print('train error cols:', [c for c in train.columns if c in ERROR_COLUMNS])

## 1. Feature list

- Take all numeric candidate columns from `feature_analysis.candidate_feature_columns` on train.
- Drop columns with <50% valid (non-NaN) rows - they would dominate the ranking with noise.
- Append the binary flags `style_cluster_is_aggressive_K2` and `style_cluster_is_aggressive_K3` from NB1 (used directly as handcrafted-composite components).
- We **do not** add `log1p / abs / abs_log1p` variants: they are monotonic transforms and do not change Spearman.

In [ ]:
EXCLUDE = set(ID_COLUMNS) | set(ERROR_COLUMNS) | {
    # NB1 bookkeeping columns we don't want ranked as "features":
    'style_label_K2', 'style_label_K3',
    'style_cluster_id_K2', 'style_cluster_id_K3',
    'style_cluster_distance_K2', 'style_cluster_distance_K3',
}

def _build_feature_frame(df):
    if df.empty:
        return df
    base = [c for c in candidate_feature_columns(df) if c not in EXCLUDE]
    base = [c for c in base if pd.to_numeric(df[c], errors='coerce').notna().mean() >= 0.5]
    out = df[base].apply(lambda s: pd.to_numeric(s, errors='coerce'))
    # Handcrafted-composite components: NB1-produced 0/1 flags.
    for K in (2, 3):
        flag = f'style_cluster_is_aggressive_K{K}'
        if flag in df.columns:
            out[flag] = pd.to_numeric(df[flag], errors='coerce').astype(float)
    return out

Xtrain = _build_feature_frame(train)
print('features_train:', Xtrain.shape)
print('first 20 features:', list(Xtrain.columns)[:20])

## 2. Scoring table (Spearman, train-only)

For each feature we compute a single **Spearman correlation** against `mtr_minfde6` on train.

- We do **not** compute Pearson (linear-only, not robust to skew) or binary-cutoff metrics
  (`top_k_capture`, `auroc_hard`, `ndcg20`, `hard_vs_easy_uplift`) because:
  - Spearman already summarises the monotonic relationship the downstream learner needs.
  - Binary-cutoff metrics all bake in an arbitrary 20% threshold and contradict each other
    under mild resampling.
  - `log1p / abs / abs_log1p` variants are monotonic transforms and do not change Spearman,
    so they were dropped.
- `valid_n` keeps rows with both score and error present. Features with <50 valid rows are skipped.

In [ ]:
def _spearman_vs_error(score: pd.Series, err: pd.Series) -> dict:
    s = pd.to_numeric(score, errors='coerce')
    e = pd.to_numeric(err,   errors='coerce')
    valid = s.notna() & e.notna()
    n = int(valid.sum())
    if n < 50:
        return {'valid_n': n, 'spearman': float('nan')}
    rho = float(s[valid].corr(e[valid], method='spearman'))
    return {'valid_n': n, 'spearman': rho}


def score_table(X: pd.DataFrame, df: pd.DataFrame, error_col: str) -> pd.DataFrame:
    if error_col not in df.columns:
        raise KeyError(f'{error_col} missing from frame')
    err = df[error_col]
    rows = []
    for feat in X.columns:
        stats = _spearman_vs_error(X[feat], err)
        if stats.get('valid_n', 0) < 50:
            continue
        stats.update({'feature': feat, 'error_col': error_col})
        rows.append(stats)
    out = pd.DataFrame(rows)
    if not out.empty:
        out = out.reindex(out['spearman'].abs().sort_values(ascending=False).index)
    return out


summary = score_table(Xtrain, train, PRIMARY_ERROR)
summary.to_csv(METRICS / 'nb2_feature_ranking_train.csv', index=False)
print('Wrote', METRICS / 'nb2_feature_ranking_train.csv', summary.shape)
print(summary.head(20).to_string(index=False))

## 3. Top-20 features vs `mtr_minfde6` (train, bar chart)

Bar chart of the top-20 features by `|Spearman|` on the train split. This is the headline per-feature diagnostic; the PASS/FAIL threshold is marked at `|Spearman| >= 0.15`.

In [ ]:
TOP_N = 20
top_n = summary.head(TOP_N).copy()
print(f'\n=== Top {TOP_N} features vs {PRIMARY_ERROR} on train ===')
print(top_n[['feature', 'spearman', 'valid_n']].to_string(index=False))

if not top_n.empty:
    fig, ax = plt.subplots(figsize=(8, 0.35 * len(top_n) + 1.5))
    order = top_n.iloc[::-1]
    ax.barh(order['feature'], order['spearman'], color='#1f77b4')
    ax.axvline( 0.15, color='red', linestyle='--', alpha=0.6, label='PASS |rho|>=0.15')
    ax.axvline(-0.15, color='red', linestyle='--', alpha=0.6)
    ax.set_xlabel(f'Spearman corr with {PRIMARY_ERROR} (train)')
    ax.legend()
    fig.tight_layout()
    fig.savefig(FIG_DIR / 'nb2_top_features.png', dpi=120, bbox_inches='tight')
    plt.show()

## 4. Handcrafted-composite components: individual Spearman

Report the individual Spearman of each of the five components that feed into the handcrafted composite `difficulty_handcrafted_v2` (used by NB3):

1. `kalman_difficulty_6s`
2. `style_cluster_is_aggressive_K2` / `style_cluster_is_aggressive_K3`
3. `trajectory_rarity_rank` (computed here from `trajectory_type` if not already in train)
4. `hard_brake_count`
5. `lateral_g_spike_count`

This table is the "ingredient Spearmans" NB3 needs to discuss whether the composite is
really greater than the sum of its parts. If NB1 did not persist `trajectory_rarity_rank`, we recompute it by ranking `trajectory_type` on train.

In [ ]:
def _traj_rarity_rank(series: pd.Series) -> pd.Series:
    """Rank `trajectory_type` so that rarer types get higher scores.

    Uses frequency on TRAIN: score(t) = 1 - freq(t), then pct-ranked.
    Monotonic with NB3's `trajectory_rarity_rank`; Spearman identical.
    """
    s = pd.to_numeric(series, errors='coerce')
    freq = s.value_counts(normalize=True)
    rarity = s.map(lambda t: 1.0 - freq.get(t, 0.0) if pd.notna(t) else np.nan)
    return rarity.rank(method='average', pct=True)


COMPONENT_COLS = [
    ('kalman_difficulty_6s',            'rank'),
    ('style_cluster_is_aggressive_K2',  'raw'),
    ('style_cluster_is_aggressive_K3',  'raw'),
    ('trajectory_rarity_rank',          'raw'),
    ('hard_brake_count',                'rank'),
    ('lateral_g_spike_count',           'rank'),
]

component_rows = []
train_err = pd.to_numeric(train[PRIMARY_ERROR], errors='coerce')
for col, mode in COMPONENT_COLS:
    if col == 'trajectory_rarity_rank' and col not in train.columns:
        if 'trajectory_type' not in train.columns:
            continue
        s = _traj_rarity_rank(train['trajectory_type'])
    elif col not in train.columns:
        component_rows.append({'component': col, 'mode': mode,
                               'spearman': float('nan'), 'valid_n': 0,
                               'note': 'column missing on train'})
        continue
    else:
        s = pd.to_numeric(train[col], errors='coerce')
        if mode == 'rank':
            s = s.rank(method='average', pct=True)
    stats = _spearman_vs_error(s, train_err)
    component_rows.append({'component': col, 'mode': mode,
                           'spearman': stats.get('spearman', float('nan')),
                           'valid_n': stats.get('valid_n', 0)})

component_df = pd.DataFrame(component_rows)
component_df.to_csv(METRICS / 'nb2_handcrafted_component_spearman.csv', index=False)
print(component_df.to_string(index=False))

## 5. Per-style-cluster error breakdown (train)

For each K, compute `n / mean / median / p90` of `mtr_minfde6` per cluster on train. This is the diagnostic that turns "is cluster X actually harder?" into a number that NB3's composite can lean on; if no cluster is meaningfully harder than the others, the style feature is not a useful learning-strategy lever.

In [ ]:
style_rows = []
for K in (2, 3):
    lbl = f'style_label_K{K}' if f'style_label_K{K}' in train.columns else f'style_cluster_id_K{K}'
    if lbl not in train.columns:
        continue
    if PRIMARY_ERROR not in train.columns:
        continue
    gr = train.groupby(lbl)[PRIMARY_ERROR].agg(
        ['size', 'mean', 'median', lambda x: np.quantile(x.dropna(), 0.90)]
    )
    gr.columns = ['n', 'mean', 'median', 'p90']
    gr['K'] = K
    gr['error'] = PRIMARY_ERROR
    style_rows.append(gr.reset_index().rename(columns={lbl: 'cluster'}))

style_err = pd.concat(style_rows, ignore_index=True) if style_rows else pd.DataFrame()
if not style_err.empty:
    style_err.to_csv(TABLES / 'nb2_style_cluster_error_breakdown_train.csv', index=False)
    print(style_err.to_string(index=False))

    fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
    for ax, K in zip(axes, (2, 3)):
        sub = style_err[style_err['K'] == K]
        if sub.empty:
            continue
        ax.bar(sub['cluster'].astype(str), sub['mean'])
        ax.set_title(f'K={K} mean {PRIMARY_ERROR} by cluster (train)')
        ax.set_ylabel(f'mean {PRIMARY_ERROR}')
        ax.set_xlabel('cluster')
    fig.tight_layout()
    fig.savefig(FIG_DIR / 'nb2_style_minfde_train.png', dpi=120, bbox_inches='tight')
    plt.show()
else:
    print('No style labels present on train; skipped.')

## 6. Verdict

PASS iff: best single feature (excluding anything in `ERROR_COLUMNS`, including `cv_*` variants) achieves `|Spearman| >= 0.15` on **train** against `mtr_minfde6`.

FAIL otherwise. A FAIL here does not by itself trigger STOP; the actual GO/PIVOT/STOP decision lives in NB3 (combinations + learned upper bound), and this verdict is only a sanity gate.

The JSON emitted to `artifacts/phase1/nb2_verdict.json` includes:

- `decision`, `best_feature`, `best_spearman`
- `top_5_features`: the top-5 features on train by `|Spearman|` (name, spearman, valid_n)
- `handcrafted_component_spearman`: one Spearman per handcrafted-composite component (see section 4), reused verbatim by NB3 and the Phase 1 report.

In [ ]:
PASS_THRESHOLD = 0.15

verdict = {
    'decision_source': 'train',
    'primary_error': PRIMARY_ERROR,
    'pass_threshold_abs_spearman': PASS_THRESHOLD,
    'decision': 'FAIL',
    'best_feature': None,
    'best_spearman': None,
    'top_5_features': [],
    'handcrafted_component_spearman': [],
}

if not summary.empty:
    top_5 = summary.head(5)[['feature', 'spearman', 'valid_n']].to_dict(orient='records')
    verdict['top_5_features'] = [
        {'feature': r['feature'],
         'spearman': float(r['spearman']),
         'valid_n': int(r['valid_n'])}
        for r in top_5
    ]
    best = summary.iloc[0].to_dict()
    verdict['best_feature']  = str(best['feature'])
    verdict['best_spearman'] = float(best['spearman'])
    if abs(best['spearman']) >= PASS_THRESHOLD:
        verdict['decision'] = 'PASS'

verdict['handcrafted_component_spearman'] = [
    {'component': r['component'],
     'mode':      r['mode'],
     'spearman':  float(r['spearman']) if pd.notna(r['spearman']) else None,
     'valid_n':   int(r['valid_n'])}
    for r in component_df.to_dict(orient='records')
]

with open(ARTIFACTS / 'nb2_verdict.json', 'w') as f:
    json.dump(verdict, f, indent=2, default=float)

print(json.dumps(verdict, indent=2, default=float))